# Module 01 — Signatures, Modules & Metrics *(interactive)*

> Three primitives you'll use in every use case: a **Signature** (typed I/O contract), a **Module** (how the model is called), and a **Metric** (quality → a number). Here's the *real* UC01 code, plus a runnable metric demo.

In [5]:
# === setup: load artifacts + define the look (run me first) ===
import json, html
from pathlib import Path
from IPython.display import HTML, display

ROOT = Path.cwd()
while not (ROOT / "scorecards").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

def load(p): return json.loads((ROOT / p).read_text(encoding="utf-8"))
def read(p): return (ROOT / p).read_text(encoding="utf-8")

PAL = dict(line="#1f2c45", tx="#e2e8f0", mut="#94a3b8", cy="#22d3ee", em="#34d399",
           vi="#a78bfa", am="#fbbf24", bl="#60a5fa", ro="#fb7185")

def _wrap(inner, pad=18):
    return (f'<div style="background:linear-gradient(180deg,#111a2e,#0f1729);color:{PAL["tx"]};'
            f'border:1px solid {PAL["line"]};border-radius:14px;padding:{pad}px;'
            f'font-family:ui-sans-serif,system-ui,Segoe UI,Roboto,sans-serif;line-height:1.55">{inner}</div>')
def show(inner): display(HTML(_wrap(inner)))
def scroll(text, h=200):
    return (f'<div style="font-size:12.5px;color:#c8d4e6;white-space:pre-wrap;max-height:{h}px;'
            f'overflow:auto;border:1px solid {PAL["line"]};border-radius:8px;padding:10px">{html.escape(str(text))}</div>')
def src(path, n=None):
    t = read(path)
    if n: t = "\n".join(t.splitlines()[:n]) + f"\n... ({len(t.splitlines())} lines total)"
    show(f'<div style="font-size:11px;color:{PAL["mut"]};letter-spacing:.06em;margin-bottom:6px">{path}</div>'
         f'<pre style="font-size:12px;color:#cbd5e1;background:#0a1426;border:1px solid {PAL["line"]};'
         f'border-radius:8px;padding:12px;max-height:340px;overflow:auto;margin:0">{html.escape(t)}</pre>')

def _bar(label, val, base, color, sub="", h=11):
    s = (f'<div style="margin:9px 0"><div style="display:flex;justify-content:space-between;font-size:13px">'
         f'<b>{label}</b><span style="color:{color};font-weight:800">{val*100:.1f}%</span></div>'
         f'<div style="position:relative;height:{h}px;background:#0c1426;border:1px solid #16223c;border-radius:6px;margin-top:4px">'
         f'<div style="position:absolute;height:100%;width:{val*100:.1f}%;border-radius:6px;background:linear-gradient(90deg,#0e7490,{color})"></div>')
    if base is not None:
        s += f'<div style="position:absolute;left:{base*100:.1f}%;top:-2px;bottom:-2px;border-left:2px dashed #93a4c4;opacity:.75"></div>'
    s += '</div>' + (f'<div style="font-size:11px;color:{PAL["mut"]};margin-top:2px">{sub}</div>' if sub else '') + '</div>'
    return s

def portfolio():
    ucs = [("ticket_router","UC01 Ticket Router"),("contract_extractor","UC02 Contract Extractor"),
           ("rag_answer","UC03 RAG Answer"),("clinical_note","UC04 Clinical Note"),("sales_email","UC05 Sales Email")]
    rows = '<div style="font-weight:700;margin-bottom:10px">Portfolio &mdash; baseline &rarr; best <span style="color:#93a4c4;font-weight:400;font-size:12px">(dashed = baseline)</span></div>'
    for f,name in ucs:
        s = load(f"scorecards/{f}.json")["summary"]
        base,best,delta = s["baseline_score"],s["best_score"],s["headline_delta"]
        win = delta>0.001; col = PAL["em"] if win else PAL["am"]
        badge = ("+%.1f%%"%(delta*100)) if win else "+0.0%"
        rows += _bar(f'{name} <span style="color:{col};font-size:12px">{badge}</span>', best, base, col,
                     f'best: {s["best_label"]}')
    show(rows)

def optimizers(uc, title=None):
    d = load(f"scorecards/{uc}.json"); s = d["summary"]; base = s["baseline_score"]
    rows = f'<div style="font-weight:700;margin-bottom:10px">{title or uc+" &mdash; every optimizer vs the baseline (dashed)"}</div>'
    for r in d["results"]:
        is_base = "baseline" in r["label"].lower(); is_best = r["label"]==s["best_label"]
        below = (not is_base) and r["score"] < base-0.001
        col = PAL["mut"] if is_base else (PAL["em"] if is_best else (PAL["ro"] if below else PAL["bl"]))
        rows += _bar(r["label"].split("(")[0].strip(), r["score"], base, col)
    show(rows)

def instr_compare(uc, label, original, title="Before (human) vs after (evolved)"):
    evolved = load(f"compiled/{uc}/{label}.json")
    pred = next(k for k in evolved if k!="metadata")
    evolved = evolved[pred]["signature"]["instructions"]
    def col(lab, text, c):
        return (f'<div style="flex:1;min-width:0"><div style="font-size:11px;color:{c};font-weight:700;'
                f'text-transform:uppercase;letter-spacing:.08em">{lab} &middot; {len(text):,} chars</div>{scroll(text,300)}</div>')
    show(f'<div style="font-weight:700;margin-bottom:8px">{title}</div>'
         f'<div style="display:flex;gap:12px;flex-wrap:wrap">{col("Original",original,PAL["mut"])}{col("Evolved",evolved,PAL["vi"])}</div>')

def demo_table(uc, label, fields, n=3, head="Demonstrations the optimizer attached"):
    demos = load(f"compiled/{uc}/{label}.json"); pred = next(k for k in demos if k!="metadata")
    demos = demos[pred]["demos"]
    cards = ""
    for d in demos[:n]:
        rowsf = ""
        for f in fields:
            v = str(d.get(f,"") or "")
            v = (v[:160]+"…") if len(v)>160 else v
            rowsf += f'<div style="font-size:12.5px;margin:2px 0"><b style="color:{PAL["cy"]}">{f}:</b> {html.escape(v)}</div>'
        cards += f'<div style="border:1px solid {PAL["line"]};border-radius:10px;padding:11px 13px;margin:8px 0;background:#0c1730">{rowsf}</div>'
    show(f'<div style="font-weight:700;margin-bottom:4px">{head} ({len(demos)} total, showing {min(n,len(demos))})</div>{cards}')

def reasoning(uc, label, fields):
    demos = load(f"compiled/{uc}/{label}.json"); pred = next(k for k in demos if k!="metadata")
    demos = demos[pred]["demos"]; aug = [d for d in demos if d.get("augmented")] or demos
    d = aug[0]
    inner = '<div style="font-weight:700;margin-bottom:6px">A teacher-bootstrapped example</div>'
    inner += f'<div style="font-size:11px;color:{PAL["mut"]};letter-spacing:.06em">REASONING TRACE</div>' + scroll(d.get("reasoning") or "(no trace)",150)
    inner += f'<div style="font-size:11px;color:{PAL["mut"]};letter-spacing:.06em;margin-top:10px">RESULT</div>'
    for f in fields:
        if d.get(f): inner += f'<div style="font-size:12.5px"><b style="color:{PAL["cy"]}">{f}:</b> {html.escape(str(d[f])[:150])}</div>'
    show(inner)

def dist(counts, title):
    mx = max(counts.values()) or 1
    rows = f'<div style="font-weight:700;margin-bottom:8px">{title}</div>'
    for k,v in sorted(counts.items(), key=lambda x:-x[1]):
        rows += _bar(k, v/mx, None, PAL["bl"], h=9).replace(f"{v/mx*100:.1f}%</span>", f"{v}</span>")
    show(rows)

def inspect(uc, label):
    prog = load(f"compiled/{uc}/{label}.json"); pred = next(k for k in prog if k!="metadata")
    sig = prog[pred].get("signature",{}); demos = prog[pred].get("demos",[]); instr = sig.get("instructions","")
    show(f'<div style="font-weight:700">{uc} &middot; {label}</div>'
         f'<div style="font-size:12px;color:{PAL["mut"]};margin:6px 0">{len(demos)} demos &middot; instruction {len(instr):,} chars</div>'
         f'{scroll(instr or "(no custom instruction)",220)}')

print("Helpers ready: portfolio() optimizers() instr_compare() demo_table() reasoning() dist() inspect() src() show()")

Helpers ready: portfolio() optimizers() instr_compare() demo_table() reasoning() dist() inspect() src() show()


## 1 · The Signature + Module (real UC01 source)

A Signature declares inputs and outputs with descriptions; a Module wraps it in a calling strategy (here, `ChainOfThought`).

In [6]:
src("use_cases/01_ticket_router/program.py")

## 2 · The Metric (real UC01 source)

The metric is the heart of optimization — it's the number the optimizer climbs. Note the **fuzzy extraction**: local models leak reasoning into outputs, so a naive string match would score correct answers as 0.

In [7]:
src("use_cases/01_ticket_router/metric.py", n=40)

## 3 · Fuzzy extraction, live (no LM needed)

Why fuzzy matching matters — run this on deliberately noisy output:

In [8]:
VALID = {"billing","engineering","sales","security"}
def extract(text, valid):
    t = text.strip().lower()
    if t in valid: return t
    first = t.split("\n")[0].strip().rstrip(".:,;")
    if first in valid: return first
    for v in valid:
        if v in t: return v
    return t

samples = ["Engineering", "Engineering\n\nThe issue is a code bug.", "billing.", "I think this is Security related"]
rows = "".join(f'<div style="font-size:13px;margin:4px 0">'
               f'<span style="color:#94a3b8">{html.escape(s[:48])!r}</span> '
               f'&rarr; <b style="color:#34d399">{extract(s, VALID)}</b></div>' for s in samples)
show("<div style=\"font-weight:700;margin-bottom:6px\">Noisy LM output &rarr; clean label</div>"+rows)

> **Takeaway:** a metric that can't handle messy real output will lie about your scores. Fuzzy extraction is one line of insurance.

Companion: [Module 01 (markdown)](../01_signatures_modules_metrics.md)